In [3]:
import pandas as pd
df = pd.read_csv("swiggy.csv")
df

,id,name,city,rating,rating_count,cost,cuisine
0,567335,AB FOODS POINT,Abohar,--,Too Few Ratings,₹ 200,"Beverages,Pizzas"
1,531342,Janta Sweet House,Abohar,4.4,50+ ratings,₹ 200,"Sweets,Bakery"
2,158203,theka coffee desi,Abohar,3.8,100+ ratings,₹ 100,Beverages
3,187912,Singh Hut,Abohar,3.7,20+ ratings,₹ 250,"Fast Food,Indian"
4,543530,GRILL MASTERS,Abohar,--,Too Few Ratings,₹ 250,"Italian-American,Fast Food"
...,...,...,...,...,...,...,...
148536,553122,The Food Delight,Yavatmal,--,Too Few Ratings,₹ 200,"Fast Food,Snacks"
148537,562647,MAITRI FOODS & BEVERAGES,Yavatmal,--,Too Few Ratings,₹ 300,Pizzas
148538,559435,Cafe Bella Ciao,Yavatmal,--,Too Few Ratings,₹ 300,"Fast Food,Snacks"
148539,418989,GRILL ZILLA,Yavatmal,--,Too Few Ratings,₹ 250,Continental


In [4]:
df.set_index("id",inplace=True)

In [5]:
df.duplicated().sum()

np.int64(35)

In [6]:
df.drop_duplicates(keep="first",inplace=True)

In [7]:
df

,name,city,rating,rating_count,cost,cuisine
id,,,,,,
567335,AB FOODS POINT,Abohar,--,Too Few Ratings,₹ 200,"Beverages,Pizzas"
531342,Janta Sweet House,Abohar,4.4,50+ ratings,₹ 200,"Sweets,Bakery"
158203,theka coffee desi,Abohar,3.8,100+ ratings,₹ 100,Beverages
187912,Singh Hut,Abohar,3.7,20+ ratings,₹ 250,"Fast Food,Indian"
543530,GRILL MASTERS,Abohar,--,Too Few Ratings,₹ 250,"Italian-American,Fast Food"
...,...,...,...,...,...,...
553122,The Food Delight,Yavatmal,--,Too Few Ratings,₹ 200,"Fast Food,Snacks"
562647,MAITRI FOODS & BEVERAGES,Yavatmal,--,Too Few Ratings,₹ 300,Pizzas
559435,Cafe Bella Ciao,Yavatmal,--,Too Few Ratings,₹ 300,"Fast Food,Snacks"


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 148506 entries, 567335 to 447770
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   name          148445 non-null  object
 1   city          148506 non-null  object
 2   rating        148445 non-null  object
 3   rating_count  148445 non-null  object
 4   cost          148400 non-null  object
 5   cuisine       148432 non-null  object
dtypes: object(6)
memory usage: 7.9+ MB


In [9]:
print("Rating values:")
print(df["rating"].value_counts().head(20))

print("\nRating count values:")
print(df["rating_count"].value_counts().head(20))

print("\nCost values:")
print(df["cost"].value_counts().head(20))

Rating values:
rating
--     87007
4       6531
4.1     6296
4.2     5821
3.8     5736
3.9     5435
4.3     5009
3.7     4253
4.4     3149
3.5     2963
3.6     2925
3.4     1879
3.3     1801
4.5     1778
4.6     1334
3.2     1202
3        859
3.1      791
4.7      648
2.8      473
Name: count, dtype: int64

Rating count values:
rating_count
Too Few Ratings    87007
20+ ratings        21635
100+ ratings       20546
50+ ratings        12009
500+ ratings        4396
1K+ ratings         2739
5K+ ratings           98
10K+ ratings          15
Name: count, dtype: int64

Cost values:
cost
₹ 200     38631
₹ 300     29700
₹ 250     19745
₹ 150     12095
₹ 400     11710
₹ 500      6378
₹ 350      6296
₹ 100      6184
₹ 600      2557
₹ 450      1456
₹ 800      1070
₹ 199      1032
₹ 120       948
₹ 1000      755
₹ 700       720
₹ 299       575
₹ 280       519
₹ 180       442
₹ 1200      392
₹ 50        351
Name: count, dtype: int64


In [10]:
import numpy as np

df["rating"] = df["rating"].replace("--", np.nan)
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

In [11]:
rating_count_mapping = {
    "Too Few Ratings": np.nan,
    "20+ ratings": 20,
    "50+ ratings": 50,
    "100+ ratings": 100,
    "500+ ratings": 500,
    "1K+ ratings": 1000,
    "5K+ ratings": 5000,
    "10K+ ratings": 10000
}

df["rating_count"] = df["rating_count"].map(rating_count_mapping)

In [12]:
df["cost"] = (
    df["cost"]
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["cost"] = pd.to_numeric(df["cost"], errors="coerce")

In [13]:
print(df[["rating", "rating_count", "cost"]].dtypes)

print("\nMissing values:")
print(df[["rating", "rating_count", "cost"]].isnull().sum())

print("\nSummary:")
print(df[["rating", "rating_count", "cost"]].describe())

rating          float64
rating_count    float64
cost            float64
dtype: object

Missing values:
rating          87068
rating_count    87068
cost              106
dtype: int64

Summary:
             rating  rating_count           cost
count  61438.000000  61438.000000  148400.000000
mean       3.894446    141.032423     287.603585
std        0.460063    332.937387     796.756557
min        1.000000     20.000000       1.000000
25%        3.700000     20.000000     200.000000
50%        4.000000     50.000000     250.000000
75%        4.200000    100.000000     300.000000
max        5.000000  10000.000000  300350.000000


In [14]:
df = df.dropna(subset=["name"]).copy()
impute = {
    "rating": df["rating"].median(),
    "rating_count": df["rating_count"].median(),
    "cost": df["cost"].median(),
    "cuisine": "Unknown"
}
df.fillna(impute,inplace=True)

In [15]:
df.isnull().sum()

name            0
city            0
rating          0
rating_count    0
cost            0
cuisine         0
dtype: int64

In [16]:
df[df["cost"] > 5000][["name", "city", "cost", "cuisine"]]

,name,city,cost,cuisine
id,,,,
456114,Oasis restaurant,"Electronic City,Bangalore",8000.0,"Biryani,North Indian"
565958,VENOM CLUB AND KITCHEN,"Hathibarkala,Dehradun",6000.0,"Italian,Indian"
524995,Aggarwal sweet india,"GTB Nagar,Delhi",5023.0,Chinese
477718,KOHINOOR HOTEL,Hinganghat,300350.0,"North Indian,Chinese"


In [17]:
print(df.shape)
print(df.dtypes)

(148445, 6)
name             object
city             object
rating          float64
rating_count    float64
cost            float64
cuisine          object
dtype: object


In [18]:
print("Number of cities:", df["city"].nunique())
print("Number of cuisines:", df["cuisine"].nunique())

Number of cities: 821
Number of cuisines: 2133


In [19]:
df["city"].value_counts().head(15)

city
Bikaner                      1666
Noida-1                      1427
Indirapuram,Delhi            1279
BTM,Bangalore                1161
Rohini,Delhi                 1134
Kothrud,Pune                 1089
Indiranagar,Bangalore        1080
Electronic City,Bangalore    1039
Greater Kailash 2,Delhi      1038
Vashi,Mumbai                 1021
Kukatpally,Hyderabad         1009
Viman Nagar,Pune             1000
sohna road,Gurgaon            976
Koramangala,Bangalore         954
Laxmi Nagar,Delhi             933
Name: count, dtype: int64

In [20]:
df["cuisine"].value_counts().head(15)

cuisine
North Indian,Chinese    6470
Indian                  6414
Chinese                 5051
North Indian            4775
Indian,Chinese          4374
South Indian            3302
Bakery                  3131
Chinese,Indian          2308
Chinese,North Indian    2288
Bakery,Desserts         2232
Biryani                 2227
Pizzas                  2215
Beverages               2156
North Indian,Indian     2004
Snacks                  1788
Name: count, dtype: int64

In [21]:
Q1 = df["cost"].quantile(0.25)
Q3 = df["cost"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)

print(df[df["cost"] > upper_limit][
    ["name", "city", "cost", "cuisine"]
].head(20))

Q1: 200.0
Q3: 300.0
Lower limit: 50.0
Upper limit: 450.0
                                    name       city    cost  \
id                                                            
573649                     PUNJABI TADKA     Abohar   700.0   
573656         PUNJABI TADKA CHICKEN HUB     Abohar   650.0   
489011                           Karim's  Adityapur   500.0   
134569                    Motel Madhuban  Adityapur   499.0   
102042                     Biryani House  Adityapur   499.0   
483552              AUROUS RESTRO LOUNGE  Adityapur   500.0   
102579     Veggie Corner( Hotel Nataraj)  Adityapur   499.0   
113036                           Kwality  Adityapur   499.0   
102581                        Chopsticks  Adityapur   499.0   
410629                          Mintelaa  Adityapur  1000.0   
537808                     The Kannelite  Adityapur   599.0   
385938                   Biryani By Kilo  Adityapur   600.0   
102631                    The White Rose  Adityapur   499.0  

In [22]:
print("Cost > ₹1,000:", (df["cost"] > 1000).sum())
print("Cost > ₹5,000:", (df["cost"] > 5000).sum())
print("Cost > ₹10,000:", (df["cost"] > 10000).sum())
print("Cost > ₹50,000:", (df["cost"] > 50000).sum())

Cost > ₹1,000: 968
Cost > ₹5,000: 4
Cost > ₹10,000: 1
Cost > ₹50,000: 1


In [23]:
df[df["cost"] > 10000][["name", "city", "cost", "cuisine"]]

,name,city,cost,cuisine
id,,,,
477718,KOHINOOR HOTEL,Hinganghat,300350.0,"North Indian,Chinese"


In [24]:
df = df[df["cost"] <= 10000].copy()

In [25]:
print(df.shape)
print(df["cost"].max())

(148444, 6)
8000.0


In [26]:
df.to_csv("cleaned_data.csv", index=True)

In [27]:
import os

print("File exists:", os.path.exists("cleaned_data.csv"))
print("Rows:", len(df))

File exists: True
Rows: 148444


In [28]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

print("\nMaximum cost:", df["cost"].max())

Shape: (148444, 6)

Missing values:
name            0
city            0
rating          0
rating_count    0
cost            0
cuisine         0
dtype: int64

Data types:
name             object
city             object
rating          float64
rating_count    float64
cost            float64
cuisine          object
dtype: object

Maximum cost: 8000.0


In [60]:
categorical_features = ["city", "cuisine"]
numerical_features = ["rating", "rating_count", "cost"]

In [30]:
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder()

In [31]:
enc = OneHotEncoder(handle_unknown="ignore",sparse_output=False)

In [61]:
enc_cat = enc.fit_transform(df[categorical_features])

In [33]:
enc_cat.shape

(148444, 2954)

In [62]:
encoded_columns = enc.get_feature_names_out(categorical_features)

print("Number of encoded columns:", len(encoded_columns))
print(encoded_columns[:20])

Number of encoded columns: 2951
['city_Abids & Koti,Hyderabad' 'city_Abohar' 'city_Adajan,Surat'
 'city_Adilabad' 'city_Adityapur' 'city_Adoni' 'city_Adyar,Chennai'
 'city_Agartala' 'city_Agra' 'city_Ahmednagar' 'city_Airoli,Mumbai'
 'city_Aizawl' 'city_Ajmer' 'city_Akola' 'city_Akota,Vadodara'
 'city_Alambagh,Lucknow' 'city_Alappuzha' 'city_Aliganj,Lucknow'
 'city_Aligarh' 'city_Alipore,Kolkata']


In [63]:
df_enc = pd.DataFrame(enc_cat,columns=encoded_columns,index=df.index)

In [64]:
enc_num = df[numerical_features].copy()
print(enc_num.shape)
print(enc_num.head())

(148056, 3)
        rating  rating_count   cost
id                                 
567335     4.0          50.0  200.0
531342     4.4          50.0  200.0
158203     3.8         100.0  100.0
187912     3.7          20.0  250.0
543530     4.0          50.0  250.0


In [65]:
encoded = pd.concat([df_enc,enc_num],axis=1)
print("Encoded shape:", encoded.shape)
print("Same number of rows:", len(df) == len(encoded))
print("Same index:", df.index.equals(encoded.index))

Encoded shape: (148056, 2954)
Same number of rows: True
Same index: True


In [67]:
print(encoded.dtypes.value_counts())

float64    2954
Name: count, dtype: int64


In [66]:
print("Non-numerical columns:")
print(encoded.select_dtypes(exclude="number").columns.tolist())

Non-numerical columns:
[]


In [68]:
encoded.to_csv("encoded_data.csv", index=True)

print("encoded_data.csv saved successfully!")

encoded_data.csv saved successfully!


In [69]:
import joblib
joblib.dump(enc, "encoder.pkl")
print("encoder.pkl saved successfully!")

encoder.pkl saved successfully!


In [70]:
import os

print("encoded_data.csv exists:", os.path.exists("encoded_data.csv"))
print("encoder.pkl exists:", os.path.exists("encoder.pkl"))

encoded_data.csv exists: True
encoder.pkl exists: True


In [71]:
from sklearn.preprocessing import MinMaxScaler

In [72]:
scaler = MinMaxScaler()
scaler_num = scaler.fit_transform(df[numerical_features])

In [73]:
scaler_num_df = pd.DataFrame(scaler_num,columns=numerical_features,index=df.index)
print(scaler_num_df.head())

        rating  rating_count      cost
id                                    
567335   0.750      0.003006  0.018868
531342   0.850      0.003006  0.018868
158203   0.700      0.008016  0.006289
187912   0.675      0.000000  0.025157
543530   0.750      0.003006  0.025157


In [74]:
recommendation = pd.concat([scaler_num_df,df_enc],axis=1)
print(recommendation.shape)
print(df.index.equals(recommendation.index))

(148056, 2954)
True


In [75]:
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [76]:
print("Minimum cost:", df["cost"].min())
print("Maximum cost:", df["cost"].max())

print("\nFiles:")
import os

for file in [
    "cleaned_data.csv",
    "encoded_data.csv",
    "encoder.pkl",
    "scaler.pkl"
]:
    print(file, ":", os.path.exists(file))

Minimum cost: 50.0
Maximum cost: 8000.0

Files:
cleaned_data.csv : True
encoded_data.csv : True
encoder.pkl : True
scaler.pkl : True


In [77]:
encoded_df = pd.read_csv("encoded_data.csv", index_col=0)

print("Cleaned shape:", df.shape)
print("Encoded shape:", encoded_df.shape)

print("Same number of rows:", len(df) == len(encoded_df))
print("Same index:", df.index.equals(encoded_df.index))

Cleaned shape: (148056, 6)
Encoded shape: (148056, 2954)
Same number of rows: True
Same index: True


In [78]:
import joblib

scaler = joblib.load("scaler.pkl")

print("Scaler:", scaler)
print("Features used by scaler:", scaler.feature_names_in_)

Scaler: MinMaxScaler()
Features used by scaler: ['rating' 'rating_count' 'cost']


In [48]:
from sklearn.metrics.pairwise import cosine_similarity

In [82]:
def recommend_restaurants(city,cuisine,rating,rating_count,cost,top_n=10):

    user_data = pd.DataFrame({"city": [city],"cuisine": [cuisine],"rating": [rating],"rating_count": [rating_count],"cost": [cost]})
    
    user_encoded = enc.transform(user_data)

    user_encoded_df = pd.DataFrame(user_encoded,columns=enc.get_feature_names_out(["city", "cuisine"]))

    user_num = scaler.transform(
        pd.DataFrame({
            "rating": [rating],
            "rating_count": [rating_count],
            "cost": [cost]
        })
    )

    user_num_df = pd.DataFrame(user_num,columns=["rating", "rating_count", "cost"])

    user_vector = pd.concat([user_num_df, user_encoded_df],axis=1)

    similarity_scores = cosine_similarity(user_vector,recommendation)[0]

    top_indices = similarity_scores.argsort()[-top_n:][::-1]

    recommendations = df.iloc[top_indices].copy()

    recommendations["similarity_score"] = (similarity_scores[top_indices])

    return recommendations

In [50]:
recommendations = recommend_restaurants(city="Bikaner",cuisine="North Indian",rating=4.0,rating_count=100,cost=300,top_n=10)

recommendations[["name", "city", "rating", "rating_count", "cost", "cuisine", "similarity_score"]]

,name,city,rating,rating_count,cost,cuisine,similarity_score
id,,,,,,,
61222,The Guls,Bikaner,4.0,50.0,300.0,North Indian,0.999995
460981,Punjabi Aanch,Bikaner,4.0,50.0,300.0,North Indian,0.999995
309545,ROYAL HAVELI,Bikaner,4.0,50.0,300.0,North Indian,0.999995
318926,Dhaba by taj,Bikaner,4.0,50.0,300.0,North Indian,0.999995
575908,Khanasutra,Bikaner,4.0,50.0,300.0,North Indian,0.999995
789,Kitchen Se,Bikaner,4.0,50.0,300.0,North Indian,0.999995
543198,POWER BOWL,Bikaner,4.0,50.0,300.0,North Indian,0.999995
176286,The champaran meat house,Bikaner,4.0,100.0,350.0,North Indian,0.999992
539554,Flavors Of Amritsar,Bikaner,4.0,50.0,350.0,North Indian,0.999987


In [52]:
import os

files = [
    "cleaned_data.csv",
    "encoded_data.csv",
    "encoder.pkl",
    "scaler.pkl"
]

for file in files:
    print(file, ":", os.path.exists(file))

cleaned_data.csv : True
encoded_data.csv : True
encoder.pkl : True
scaler.pkl : True


In [54]:
print(df["cost"].describe())
print()
print(df["cost"].value_counts().head(20))

count    148444.000000
mean        285.570801
std         167.581581
min           1.000000
25%         200.000000
50%         250.000000
75%         300.000000
max        8000.000000
Name: cost, dtype: float64

cost
200.0     38631
300.0     29700
250.0     19790
150.0     12095
400.0     11710
500.0      6378
350.0      6296
100.0      6184
600.0      2557
450.0      1456
800.0      1070
199.0      1032
120.0       948
1000.0      755
700.0       720
299.0       575
280.0       519
180.0       442
1200.0      392
50.0        351
Name: count, dtype: int64


In [55]:
print(df[["name", "city", "rating", "rating_count", "cost", "cuisine"]].head(20))

                                           name    city  rating  rating_count  \
id                                                                              
567335                           AB FOODS POINT  Abohar     4.0          50.0   
531342                        Janta Sweet House  Abohar     4.4          50.0   
158203                        theka coffee desi  Abohar     3.8         100.0   
187912                                Singh Hut  Abohar     3.7          20.0   
543530                            GRILL MASTERS  Abohar     4.0          50.0   
158204                                Sam Uncle  Abohar     3.6          20.0   
156588                         shere punjab veg  Abohar     4.0         100.0   
244866                Shri Balaji Vaishno Dhaba  Abohar     4.0          50.0   
156602                 Hinglaj Kachori Bhandhar  Abohar     4.2          20.0   
158193                                yummy hub  Abohar     4.0          50.0   
407249             CHAWLA SA

In [56]:
print("Cost minimum:", df["cost"].min())
print("Cost maximum:", df["cost"].max())
print("Unique costs:", df["cost"].nunique())

Cost minimum: 1.0
Cost maximum: 8000.0
Unique costs: 362


In [57]:
df[df["cost"] < 50][
    ["name", "city", "rating", "rating_count", "cost", "cuisine"]
].sort_values("cost").head(30)

,name,city,rating,rating_count,cost,cuisine
id,,,,,,
543724,Taau's special restaurant,Wardha,4.0,50.0,1.0,"Fast Food,Chinese"
486351,Poonam Hotel,Sikar,4.0,50.0,1.0,"North Indian,Beverages"
444871,Sweet Bite,"PCMC,Pune",4.0,50.0,1.0,Bakery
559198,UTTAM SWEETS,"Narhe,Pune",4.0,50.0,1.0,Sweets
469187,JAGDAMBA RESTAURANT,"Magarpatta,Pune",2.9,20.0,1.0,"Indian,Chinese"
524495,Sapna Momos,"Viman Nagar,Pune",4.0,50.0,1.0,Chinese
412032,HOTEL CHIGURU,"Battarahalli,Bangalore",4.0,50.0,1.0,"Chinese,South Indian"
457866,Shine family restaurant,"Basaveshwaranagar,Bangalore",4.0,50.0,1.0,"Chinese,Indian"
458490,Chinni's Biryani,"Sanjay Nagar, New BEL Road,Bangalore",4.0,50.0,1.0,"Biryani,Chinese"


In [58]:
df = df[
    (df["cost"] >= 50) &
    (df["cost"] <= 10000)
].copy()

print("Shape after cost cleaning:", df.shape)
print("Minimum cost:", df["cost"].min())
print("Maximum cost:", df["cost"].max())

Shape after cost cleaning: (148056, 6)
Minimum cost: 50.0
Maximum cost: 8000.0


In [59]:
print("Remaining rows:", len(df))

Remaining rows: 148056


In [80]:
results = recommend_restaurants(
    city="Bikaner",
    cuisine="North Indian",
    rating=4.0,
    rating_count=100,
    cost=300,
    top_n=10
)

print(
    results[
        [
            "name",
            "city",
            "rating",
            "rating_count",
            "cost",
            "cuisine",
            "similarity"
        ]
    ].to_string(index=False)
)

MemoryError: Unable to allocate 3.26 GiB for an array with shape (2954, 148056) and data type float64